# Scaling — season style tables

Thin driver for the Phase 0 batch runner (`src/features/batch.py`). Builds the two
backbone tables for one season and persists them to `data/features/`:

- **`match_team_style`** — one row per `(match_id, team)`; per-match style fingerprint + context covariates.
- **`possession_chains`** — `build_possession_summary` concatenated across the season, tagged with context.

Swapping seasons is a one-line change to `SEASON` (registry-driven — opponent strength is a static
final-table lookup for partial-coverage seasons like 2003/04, computed standings for full-league seasons).

In [3]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "src" / "config.py").is_file()
)
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import FEATURES_DIR
from src.features.batch import build_season_features

In [4]:
SEASON = "2003/2004"

style, chains = build_season_features(SEASON)
print(f"match_team_style : {style.shape[0]} rows x {style.shape[1]} cols")
print(f"possession_chains: {chains.shape[0]} rows x {chains.shape[1]} cols")
style.head()

match_team_style : 76 rows x 152 cols
possession_chains: 8241 rows x 52 cols


,season,match_id,date,match_week,team,opponent,venue,goals_for,goals_against,result,...,through_balls_per100pass,passes_into_final_third_per100pass,carries_per100pass,dribbles_per100pass,dribbles_complete_per100pass,counterpresses_per100pass,duels_per100pass,fouls_committed_per100pass,shots_per100pass,xg_per_possession
0,2003/2004,3749493,2003-08-16 13:00:00.000,1,Arsenal,Everton,home,2,1,W,...,0.000000,30.094787,81.042654,6.872038,3.080569,8.767773,6.635071,2.606635,3.317536,0.020201
1,2003/2004,3749493,2003-08-16 13:00:00.000,1,Everton,Arsenal,away,1,2,L,...,0.787402,49.343832,61.679790,1.837270,1.312336,20.472441,14.435696,4.986877,3.937008,0.013203
2,2003/2004,3749448,2004-04-09 11:30:00.000,31,Arsenal,Liverpool,home,4,2,W,...,0.000000,26.356589,70.930233,3.294574,2.131783,8.527132,7.558140,3.682171,2.906977,0.021117
3,2003/2004,3749448,2004-04-09 11:30:00.000,31,Liverpool,Arsenal,away,2,4,L,...,0.262467,36.220472,62.992126,3.937008,2.624672,12.335958,7.874016,3.674541,3.412073,0.018661
4,2003/2004,3749196,2003-11-08 14:00:00.000,12,Arsenal,Tottenham Hotspur,home,2,1,W,...,0.212766,35.319149,65.531915,2.978723,2.553191,10.000000,8.297872,2.340426,2.978723,0.015160


## Validation gate

The scaled runner must reproduce the single-match case study (Arsenal 2–1 Everton, `match_id` 3749493).
`poss`/`rhythm` totals are summed across both teams (matching the case-study figures).

In [5]:
if SEASON == "2003/2004":
    gate = style[style["match_id"] == 3749493]
    ars = gate[gate["team"] == "Arsenal"].iloc[0]

    assert int(ars["passes"]) == 422, ars["passes"]
    assert int(gate["poss_possessions"].sum()) == 201
    assert int(gate["rhythm_possessions"].sum()) == 70
    assert ars["venue"] == "home" and ars["opponent"] == "Everton" and ars["result"] == "W"
    assert (int(ars["opp_final_position"]), int(ars["opp_final_points"])) == (17, 39)

    print("validation gate passed ✓")
    print(f"  Arsenal passes           : {int(ars['passes'])}")
    print(f"  both-teams possessions   : {int(gate['poss_possessions'].sum())}")
    print(f"  both-teams rhythm chains : {int(gate['rhythm_possessions'].sum())}")
else:
    print(f"gate skipped (only defined for 2003/2004; SEASON={SEASON})")

validation gate passed ✓
  Arsenal passes           : 422
  both-teams possessions   : 201
  both-teams rhythm chains : 70


## Sanity checks

Rows should be `matches × 2`, ranges should look like real football, and no column should be all-NaN.
Every possession chain should carry its context tags (0 missing).

In [6]:
print("matches:", style["match_id"].nunique(), "| rows:", len(style), "(expect matches x 2)")
print()
for col in ["pass_completion_pct", "avg_pass_length_m", "poss_possessions", "xg"]:
    print(f"  {col:22s}: {style[col].min():6.1f} .. {style[col].max():6.1f}")

all_nan = [c for c in style.columns if style[c].isna().all()]
print("\nall-NaN columns:", all_nan or "none")
print("possession_chains missing context:", int(chains["season"].isna().sum()))

matches: 38 | rows: 76 (expect matches x 2)

  pass_completion_pct   :   57.1 ..   88.0
  avg_pass_length_m     :   18.3 ..   29.2
  poss_possessions      :   86.0 ..  133.0
  xg                    :    0.1 ..    3.6

all-NaN columns: none
possession_chains missing context: 0


## Persist

`match_team_style` as CSV (small, human-inspectable); `possession_chains` as parquet (large, fast).

In [7]:
style.to_csv(FEATURES_DIR / "match_team_style.csv", index=False)
chains.to_parquet(FEATURES_DIR / "possession_chains.parquet")
print("wrote:")
print(" ", FEATURES_DIR / "match_team_style.csv")
print(" ", FEATURES_DIR / "possession_chains.parquet")

wrote:
  /Users/sothea/Documents/Files/Code/prem analysis/data/features/match_team_style.csv
  /Users/sothea/Documents/Files/Code/prem analysis/data/features/possession_chains.parquet
